In [68]:
import pandas as pd
import numpy as np
import re
import os
import joblib
from datetime import datetime

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

Загрузка данных и вывод

In [69]:
from google.colab import drive
drive.mount('/content/drive')

train = pd.read_parquet("/content/drive/MyDrive/tochkabank/train.parquet")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [70]:
train.iloc[0]


,dff0d182-5434-46e2-9183-be278f66667f
text,"Соцветия ромашки, которые продаются в аптеках,..."
integrity,1.0
integrity_reasoning,The content is an informative text about the u...
factuality,1.0
factuality_reasoning,The content provides informative and fact-base...
truthfulness,1.0
truthfulness_reasoning,The content provides credible information abou...


Работаем с integrity

In [71]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15000 entries, dff0d182-5434-46e2-9183-be278f66667f to a4449af6-c7ef-4e8f-abb6-97ed094da5fe
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   text                    15000 non-null  object 
 1   integrity               15000 non-null  float64
 2   integrity_reasoning     15000 non-null  object 
 3   factuality              15000 non-null  float64
 4   factuality_reasoning    15000 non-null  object 
 5   truthfulness            15000 non-null  float64
 6   truthfulness_reasoning  15000 non-null  object 
dtypes: float64(3), object(4)
memory usage: 937.5+ KB


In [72]:
train_integrity = train[train["integrity"] != 0.5].copy()
train_integrity = train_integrity[["text", "integrity"]]

train_integrity["integrity"].value_counts(normalize=True)
##train_integrity.info()

,proportion
integrity,
1.0,0.762877
0.0,0.237123


In [77]:
train_integrity

,text,integrity
uuid,,
dff0d182-5434-46e2-9183-be278f66667f,"Соцветия ромашки, которые продаются в аптеках,...",1.0
8268f315-03db-4f12-aa46-0b968c3b1b19,Кто из черниговцев сам будет убирать придомову...,1.0
dc7cd0dd-9eca-418e-8356-9050c4a17cdc,Тамбовчане смогут услышать романсы 20-30 годов...,1.0
e9f43939-22b1-4a4e-ab5a-34a25b269039,Человек может отказаться от ТВ и проигрывателе...,1.0
412b31bb-ba09-44ae-abd7-49c9b783b005,Простая интеграция в системы PROFINET®\nВозмож...,1.0
...,...,...
13b93d68-1dd4-49c9-b1d9-310e42633370,Ключевые результаты за 1 квартала 2020 года:\n...,1.0
82f28832-78f7-4b81-bd89-af515d072c5d,Диван SleepArt Алтгерия по цене 66 000 руб. с ...,1.0
254cb352-1fa4-4847-9f42-b85bbadc6250,Так что ТС в таких странах будет бесполезна.\n...,1.0


Добавляем фичи

In [73]:
import numpy as np
import re

def extract_features(text):
    words = text.split()
    sentences = re.split(r'[.!?]+', text)

    num_chars = len(text)
    num_words = len(words)
    num_sentences = len([s for s in sentences if s.strip()])

    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    avg_sentence_len = num_words / num_sentences if num_sentences else 0
    unique_ratio = len(set(words)) / num_words if num_words else 0

    digit_ratio = sum(c.isdigit() for c in text) / num_chars if num_chars else 0
    upper_ratio = sum(c.isupper() for c in text) / num_chars if num_chars else 0
    newline_count = text.count("\n")

    return [
        num_chars,
        num_words,
        num_sentences,
        avg_word_len,
        avg_sentence_len,
        unique_ratio,
        digit_ratio,
        upper_ratio,
        newline_count
    ]


In [8]:
X = train_integrity["text"]
y = train_integrity["integrity"]

numeric_features = np.array(
    [extract_features(text) for text in tqdm(X)]
)

print("Numeric features shape:", numeric_features.shape)

scaler = StandardScaler()
numeric_features_scaled = scaler.fit_transform(numeric_features)

  0%|          | 0/14600 [00:00<?, ?it/s]

Numeric features shape: (14600, 9)


# **tfidf**

In [97]:


tfidf = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)
tfidf.fit_transform(train["text"])
X_tfidf = tfidf.transform(X)

print("TF-IDF shape:", X_tfidf.shape)


TF-IDF shape: (14600, 50000)


# Обединяем получаем  X_full

In [98]:
X_full = hstack([X_tfidf, numeric_features_scaled]).tocsr()
print("Final feature matrix:", X_full.shape)

Final feature matrix: (14600, 50009)


# Излишняя крос-валидация


```


skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


f1_scores = []

for train_idx, val_idx in skf.split(X_full, y):

    X_train, X_val = X_full[train_idx], X_full[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LogisticRegression(

        
        solver="saga",
       
        max_iter=1200,
        n_jobs=-1,
        
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)

    print(f"\nFold  F1: {f1:.4f}")

print("Mean F1:", np.mean(f1_scores))
```



# Работаем со всем набором данных смотрим итоговый F1

In [99]:



X_train, X_val, y_train, y_val = train_test_split(
    X_full, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


model = LogisticRegression(
    solver="saga",
    max_iter=1500,
    n_jobs=-1,

)


model.fit(X_train, y_train)


y_val_pred = model.predict(X_val)
f1_val = f1_score(y_val, y_val_pred)
print("F1 на val:", f1_val)





F1 на val: 0.8670861187573732


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


# Сохраняем модель, если необходимо

In [38]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs("models", exist_ok=True)
model_name = f"models/integrity_{timestamp}_{f1_val}.pkl"
joblib.dump(model, model_name)

print(f"Saved: {model_name}")

Saved: models/integrity_20260220_064914_0.8677685950413223.pkl


# Результаты крос валидации

Mean F1: 0.8672834755030792
model = LogisticRegression(
        solver="saga",
        max_iter=1000,
        n_jobs=-1
    )

# 2 часть

In [100]:
train_fact = train[train["factuality"] != 0.5].copy()

X_text = train_fact["text"]
y_fact = train_fact["factuality"].astype(int)

X_text.info()
train_fact["factuality"].value_counts(normalize=True)

<class 'pandas.core.series.Series'>
Index: 13510 entries, dff0d182-5434-46e2-9183-be278f66667f to a4449af6-c7ef-4e8f-abb6-97ed094da5fe
Series name: text
Non-Null Count  Dtype 
--------------  ----- 
13510 non-null  object
dtypes: object(1)
memory usage: 211.1+ KB


,proportion
factuality,
1.0,0.702591
0.0,0.297409


In [101]:
import re
import numpy as np

def extract_features_fact(text):
    words = text.split()
    num_chars = len(text)
    num_words = len(words)

    digit_ratio = sum(c.isdigit() for c in text) / num_chars if num_chars else 0
    upper_ratio = sum(c.isupper() for c in text) / num_chars if num_chars else 0

    url_count = len(re.findall(r'http\S+', text))
    percent_count = text.count('%')
    year_count = len(re.findall(r'\b(19|20)\d{2}\b', text))
    number_count = len(re.findall(r'\d+', text))

    unique_ratio = len(set(words)) / num_words if num_words else 0

    return [
        num_chars,
        num_words,
        digit_ratio,
        upper_ratio,
        url_count,
        percent_count,
        year_count,
        number_count,
        unique_ratio
    ]

In [102]:
from tqdm.notebook import tqdm

numeric_fact = np.array(
    [extract_features_fact(text) for text in tqdm(X_text)]
)

  0%|          | 0/13510 [00:00<?, ?it/s]

In [103]:
from sklearn.feature_extraction.text import TfidfVectorizer

"""
tfidf_fact = TfidfVectorizer(
    max_features=45000,
    ngram_range=(1,2),
    min_df=3
)"""

X_tfidf_fact = tfidf.transform(X_text)
X_tfidf_fact

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3290945 stored elements and shape (13510, 50000)>

In [104]:
from sklearn.preprocessing import StandardScaler

scaler_fact = StandardScaler()
numeric_fact_scaled = scaler_fact.fit_transform(numeric_fact)

In [105]:
from scipy.sparse import hstack

X_full_fact = hstack([X_tfidf_fact, numeric_fact_scaled]).tocsr()

In [124]:
X_full_fact

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3412535 stored elements and shape (13510, 50009)>

# Крос-валидация


```
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_idx, val_idx in skf.split(X_full_fact, y_fact):

    X_train, X_val = X_full_fact[train_idx], X_full_fact[val_idx]
    y_train, y_val = y_fact.iloc[train_idx], y_fact.iloc[val_idx]

    model_fact = LogisticRegression(
       solver="saga",
       max_iter=2000,
       class_weight="balanced",
       C=0.5,
       n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)

    print(f"\nFold  F1: {f1:.4f}")

print("Mean F1:", np.mean(f1_scores))

```



# Результат крос-валидации
Mean F1: 0.8586725672534629

In [125]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full_fact, y_fact,
    test_size=0.2,
    stratify=y_fact,
    random_state=42
)

model_fact = LogisticRegression(
       solver="saga",
       max_iter=2000,
       class_weight="balanced",
       C=0.7,
       n_jobs=-1
    )


model_fact.fit(X_train, y_train)


y_val_pred = model_fact.predict(X_val)
f1_val = f1_score(y_val, y_val_pred)
print("F1 на val:", f1_val)


F1 на val: 0.870722433460076


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Сохраение модели при необходимости

In [108]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs("models", exist_ok=True)
model_name = f"models/factuality_{timestamp}_{f1_val}.pkl"
joblib.dump(model, model_name)

print(f"Saved: {model_name}")

Saved: models/factuality_20260220_083228_0.870722433460076.pkl


3 часть

In [109]:
train_truth = train[train["truthfulness"] != 0.5].copy()

X_text = train_truth["text"]
y_truth = train_truth["truthfulness"].astype(int)

train_truth["truthfulness"].value_counts(normalize=True)

,proportion
truthfulness,
1.0,0.900159
0.0,0.099841


# Доп фичи

In [110]:
import re
import numpy as np

uncertain_words = [
    "maybe", "perhaps", "possibly",
    "likely", "unlikely", "allegedly",
    "rumor", "rumour", "claim", "claimed"
]

def extract_features_truth(text):
    words = text.lower().split()
    num_chars = len(text)
    num_words = len(words)

    exclam_count = text.count("!")
    question_count = text.count("?")

    upper_ratio = sum(c.isupper() for c in text) / num_chars if num_chars else 0
    digit_ratio = sum(c.isdigit() for c in text) / num_chars if num_chars else 0

    uncertain_count = sum(word in words for word in uncertain_words)

    unique_ratio = len(set(words)) / num_words if num_words else 0

    return [
        num_chars,
        num_words,
        exclam_count,
        question_count,
        upper_ratio,
        digit_ratio,
        uncertain_count,
        unique_ratio
    ]

In [111]:
from tqdm.notebook import tqdm

numeric_truth = np.array(
    [extract_features_truth(text) for text in tqdm(X_text)]
)

  0%|          | 0/10056 [00:00<?, ?it/s]

In [112]:
from sklearn.feature_extraction.text import TfidfVectorizer

'''
tfidf_truth = TfidfVectorizer(
    max_features=40000,
    ngram_range=(1,2),
    min_df=3
)'''

X_tfidf_truth = tfidf.transform(X_text)

In [113]:
from sklearn.preprocessing import StandardScaler

scaler_truth = StandardScaler()
numeric_truth_scaled = scaler_truth.fit_transform(numeric_truth)

# Объединение

In [114]:
from scipy.sparse import hstack

X_full_truth = hstack([X_tfidf_truth, numeric_truth_scaled]).tocsr()

# Крос-валидация


```
from sklearn.linear_model import LogisticRegression


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_idx, val_idx in skf.split(X_full_truth, y_truth):

    X_train, X_val = X_full_truth[train_idx], X_full_truth[val_idx]
    y_train, y_val = y_truth.iloc[train_idx], y_truth.iloc[val_idx]

    model_fact = LogisticRegression(
       solver="saga",
       max_iter=2000,
       class_weight="balanced",
       C=0.5,
       n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)

    print(f"\nFold  F1: {f1:.4f}")

print("Mean F1:", np.mean(f1_scores))
```



# Результат крос валидации
Mean F1: 0.9328273398134671

In [138]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full_truth, y_truth,
    test_size=0.2,
    stratify=y_truth,
    random_state=42
)

model_truth = LogisticRegression(
       solver="saga",
       max_iter=2000,
       class_weight="balanced",
       C=0.5,
       n_jobs=-1
    )


model_truth.fit(X_train, y_train)


y_val_pred = model_truth.predict(X_val)
f1_val = f1_score(y_val, y_val_pred)
print("F1 на val:", f1_val)




F1 на val: 0.9317592859199539


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Сохранение, если необходимо

In [94]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs("models", exist_ok=True)
model_name = f"models/truthfulness_{timestamp}_{f1_val}.pkl"
joblib.dump(model, model_name)

print(f"Saved: {model_name}")

Saved: models/truthfulness_20260220_081434_0.932681242807825.pkl


# Запуск на test и создание submission.csv

In [95]:

test = pd.read_parquet("/content/drive/MyDrive/tochkabank/test.parquet")

In [142]:
test.head()

,text
uuid,
61f42863-9b53-4e78-a952-714da49f0c7a,В Вахрушах началось газоснабжение Укрзализныця...
f07bb53f-5639-46fe-9741-9465d516b8d4,"«Соглашение было достигнуто, но…» Фонсека – о ..."
f1d3d72b-3432-4604-8146-15a7d94b1598,Принципиальная электрическая схема компьютера ...
935d8a6c-6d75-4321-a66d-f3061d267df7,я добавлю - проехать в павильон пригородных ка...
c7f7f912-59f6-4db3-abc3-510f9318a3ff,Министерство образования и науки РД — О провед...


In [96]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 61f42863-9b53-4e78-a952-714da49f0c7a to 9fddf7bf-759b-40ea-836b-d8296c074639
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5000 non-null   object
dtypes: object(1)
memory usage: 78.1+ KB


In [116]:
X_test_tfidf = tfidf.transform(test["text"])

# 1) единство темы (integrity):

In [128]:
numeric_test_integr = np.array(
    [extract_features(text) for text in tqdm(test["text"])]
)
numeric_test_integr = scaler.transform(numeric_test_integr)
from scipy.sparse import hstack

X_test_integr = hstack([X_test_tfidf, numeric_test_integr])
y_pred_integr = model.predict(X_test_integr)

  0%|          | 0/5000 [00:00<?, ?it/s]

# 2) фактологичность (factuality):

In [129]:
numeric_test_fact = np.array(
    [extract_features_fact(text) for text in tqdm(test["text"])]
)

  0%|          | 0/5000 [00:00<?, ?it/s]

In [134]:
numeric_test_fact = scaler_fact.transform(numeric_test_fact)

In [135]:
from scipy.sparse import hstack

X_test_full = hstack([X_test_tfidf, numeric_test_fact])

In [136]:
y_pred_full = model_fact.predict(X_test_full)

# 3) правдивость (truthfulness):

In [139]:
numeric_test_truth = np.array(
    [extract_features_truth(text) for text in tqdm(test["text"])]
)
numeric_test_truth = scaler_truth.transform(numeric_test_truth)
from scipy.sparse import hstack

X_test_truth = hstack([X_test_tfidf, numeric_test_truth])
y_pred_truth = model_truth.predict(X_test_truth)


  0%|          | 0/5000 [00:00<?, ?it/s]

In [148]:
submission = pd.DataFrame({
    "uuid": test.index,
    "integrity": y_pred_integr.astype('int'),
    "truthfulness": y_pred_full,
    "factuality": y_pred_truth
})

In [149]:
submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   uuid          5000 non-null   object
 1   integrity     5000 non-null   int64 
 2   truthfulness  5000 non-null   int64 
 3   factuality    5000 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 156.4+ KB


In [153]:
submission["factuality"].value_counts(normalize=True)

,proportion
factuality,
1,0.7468
0,0.2532


In [154]:
submission.to_csv("submission.csv", index=False)

В решении задачи использовал TF-IDF и LogisticRegression
  TF-IDF использовался так как
  LogisticRegression использовался так как

*   TF-IDF использовался так как он формирует матрицу высокой размерности, где тексты становятся линейно разделимыми
*   LogisticRegression использовался так как хорошо работает с высокоразмерными разреженными признаками. Быстрее и устойчивее по сравнению с бустингом и нейросетями

Для дороботки и улучшения необходимо дополнительно учитывать значения 0.5 и определять его в отдельный класс выделяя признаки. (нехватило времени)
